
# Reliability-Aware Hypergraph Attention Network Based Risk-Aware Multi-Agent Scheduling for Lifetime Maximization in Wireless Sensor Networks

### Complete end-to-end implementation

This notebook implements the proposed pipeline:

**WSN Data Collection → Preprocessing → DRNR → Dynamic Weighted Hypergraph → RA-HAN → Mamba-2 Temporal Prediction → RAMS/PPO → Adaptive Sleep-Wake + Routing → Closed-Loop Updating → Lifetime/Energy/Reliability Evaluation**

The notebook is intentionally self-contained. When real WSN/NS-3/OMNeT++ traces are not available, it generates a reproducible temporal WSN simulation dataset containing:

- Node coordinates and sink distance
- Initial/residual energy
- Energy consumption
- RSSI and link quality
- Packet generation / traffic load
- Packet loss and retransmissions
- Latency
- Node degree / neighborhood stability
- Failure/dead-node information
- Previous scheduling decisions

The learning components are implemented with PyTorch. The temporal block is a **CPU-safe Mamba-2-style selective state-space implementation**, with an optional `mamba-ssm` installation path commented in the setup cell. This avoids CUDA/version failures commonly encountered in Colab while preserving the selective state-space modeling role required by the method.


In [ ]:

# ============================================================
# CELL 1 — INSTALL / IMPORTS / REPRODUCIBILITY
# ============================================================
!pip -q install numpy pandas scikit-learn matplotlib seaborn openpyxl

import os, gc, math, random, time, warnings, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


In [ ]:

# ============================================================
# CELL 2 — CONFIGURATION
# ============================================================
CONFIG = {
    "n_nodes": 30,
    "n_steps": 240,
    "history": 12,
    "prediction_horizon": 3,
    "area_size": 100.0,
    "sink_x": 50.0,
    "sink_y": 50.0,
    "initial_energy": 100.0,
    "radio_range": 42.0,
    "packet_bits": 4000,
    "base_traffic": 0.06,
    "hyperedge_k": 5,
    "n_heads": 4,
    "hidden_dim": 64,
    "temporal_dim": 64,
    "dropout": 0.10,
    "batch_size": 64,
    "epochs_prediction": 12,
    "epochs_ppo": 15,
    "ppo_steps_per_epoch": 64,
    "ppo_clip": 0.20,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "ppo_lr": 3e-4,
    "max_episode_steps": 120,
}

OUT_DIR = "/content/RA_HAN_RAMS_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

print(json.dumps(CONFIG, indent=2))


## Stage 1 — Wireless Sensor Network Data Collection / Simulation

In [ ]:

# ============================================================
# CELL 3 — WSN SIMULATOR
# ============================================================
class WSNEnvironment:
    def __init__(self, cfg, seed=42):
        self.cfg = cfg
        self.rng = np.random.default_rng(seed)
        self.N = cfg["n_nodes"]
        self.T = cfg["n_steps"]
        self.area = cfg["area_size"]
        self.radio_range = cfg["radio_range"]
        self.sink = np.array([cfg["sink_x"], cfg["sink_y"]], dtype=np.float32)

        self.coords = self.rng.uniform(0, self.area, size=(self.N, 2)).astype(np.float32)
        self.initial_energy = np.full(self.N, cfg["initial_energy"], dtype=np.float32)

    def distances(self):
        diff = self.coords[:, None, :] - self.coords[None, :, :]
        return np.sqrt((diff ** 2).sum(axis=-1))

    def generate(self):
        D = self.distances()
        np.fill_diagonal(D, np.inf)
        neigh = D <= self.radio_range

        rows = []
        energy = self.initial_energy.copy()
        prev_active = np.ones(self.N, dtype=np.float32)

        for t in range(self.T):
            degree = neigh.sum(axis=1).astype(float)
            neighborhood_stability = np.clip(
                0.88 + 0.08*np.sin(t/17.0) + self.rng.normal(0, 0.025, self.N),
                0.45, 0.99
            )

            # Traffic varies temporally and spatially.
            traffic = np.clip(
                self.cfg["base_traffic"] *
                (1 + 0.35*np.sin(t/13.0)) *
                self.rng.lognormal(mean=0, sigma=0.18, size=self.N),
                0.005, 0.35
            )

            dist_sink = np.linalg.norm(self.coords - self.sink, axis=1)
            rssi = -35 - 0.42*dist_sink + self.rng.normal(0, 2.0, self.N)

            # Link quality decreases with distance, traffic and local instability.
            lq = np.clip(
                0.97*np.exp(-dist_sink/155.0)
                * (1 - 0.45*traffic)
                * neighborhood_stability
                + self.rng.normal(0, 0.025, self.N),
                0.05, 0.995
            )

            packet_loss = np.clip(1-lq + self.rng.normal(0, .01, self.N), 0, .95)
            retrans = np.round(traffic*12*(1-lq) + self.rng.poisson(0.15, self.N)).astype(int)
            latency = np.clip(
                8 + 0.22*dist_sink + 35*packet_loss + 3*traffic +
                self.rng.normal(0, 1.5, self.N), 1, 120
            )

            tx_energy = 0.018 + 0.00012*(dist_sink**2) + 0.004*traffic
            rx_energy = 0.008 + 0.002*degree
            sleep_factor = 0.25 + 0.75*prev_active
            energy_consumption = (tx_energy + rx_energy) * traffic * 3.0 * sleep_factor

            # Stress events make failures realistic.
            stress = (self.rng.random(self.N) < 0.012).astype(float)
            energy_consumption *= (1 + 2.0*stress)

            energy = np.maximum(energy - energy_consumption, 0)

            # Failure probability depends on low energy + poor reliability.
            failure_prob = np.clip(
                0.02
                + 0.35*(1-energy/self.cfg["initial_energy"])
                + 0.40*(1-lq)
                + 0.08*(traffic)
                + 0.12*stress,
                0, 0.98
            )

            dead = (energy <= 0).astype(int)
            failure = ((self.rng.random(self.N) < failure_prob) | (dead == 1)).astype(int)

            for i in range(self.N):
                rows.append({
                    "time": t,
                    "node_id": i,
                    "x": self.coords[i,0],
                    "y": self.coords[i,1],
                    "initial_energy": self.initial_energy[i],
                    "residual_energy": energy[i],
                    "energy_consumption": energy_consumption[i],
                    "tx_energy": tx_energy[i],
                    "rx_energy": rx_energy[i],
                    "traffic_load": traffic[i],
                    "distance_to_sink": dist_sink[i],
                    "rssi": rssi[i],
                    "link_quality": lq[i],
                    "packet_loss_rate": packet_loss[i],
                    "retransmissions": retrans[i],
                    "latency_ms": latency[i],
                    "node_degree": degree[i],
                    "neighborhood_stability": neighborhood_stability[i],
                    "failure_probability_true": failure_prob[i],
                    "failure": failure[i],
                    "dead": dead[i],
                    "previous_active": prev_active[i],
                })

            prev_active = np.where(dead > 0, 0, prev_active)

        return pd.DataFrame(rows), D, neigh, self.coords

sim = WSNEnvironment(CONFIG, seed=SEED)
raw_df, distance_matrix, adjacency, coordinates = sim.generate()

raw_path = os.path.join(OUT_DIR, "raw_wsn_temporal_dataset.csv")
raw_df.to_csv(raw_path, index=False)

print("Raw shape:", raw_df.shape)
print(raw_df.head())
print("Saved:", raw_path)


## Stage 2 — Robust Scaling, Imputation, Temporal Windowing and Outlier Handling

In [ ]:

# ============================================================
# CELL 4 — PREPROCESSING + TEMPORAL WINDOWS
# ============================================================
NODE_FEATURES = [
    "residual_energy", "energy_consumption", "tx_energy", "rx_energy",
    "traffic_load", "distance_to_sink", "rssi", "link_quality",
    "packet_loss_rate", "retransmissions", "latency_ms", "node_degree",
    "neighborhood_stability", "previous_active"
]

TARGET_FEATURES = [
    "residual_energy", "energy_consumption",
    "failure_probability_true", "link_quality"
]

df = raw_df.copy()

# Deliberate missing values to exercise the imputation pipeline.
rng = np.random.default_rng(SEED)
mask = rng.random(len(df)) < 0.008
df.loc[mask, "rssi"] = np.nan
mask2 = rng.random(len(df)) < 0.006
df.loc[mask2, "link_quality"] = np.nan

imputer = SimpleImputer(strategy="median")
df[NODE_FEATURES] = imputer.fit_transform(df[NODE_FEATURES])

# Robust clipping / scaling.
for c in NODE_FEATURES:
    q1, q3 = df[c].quantile([0.01, 0.99])
    df[c] = df[c].clip(q1, q3)

scaler = RobustScaler()
df[NODE_FEATURES] = scaler.fit_transform(df[NODE_FEATURES])

# Restore targets in a separate unscaled table for evaluation.
target_raw = raw_df[TARGET_FEATURES].copy()

# Build [time, node, feature] tensors.
T = CONFIG["n_steps"]
N = CONFIG["n_nodes"]
Fdim = len(NODE_FEATURES)

X_np = df[NODE_FEATURES].values.reshape(T, N, Fdim).astype(np.float32)
Y_raw_np = raw_df[TARGET_FEATURES].values.reshape(T, N, len(TARGET_FEATURES)).astype(np.float32)
failure_np = raw_df["failure"].values.reshape(T, N).astype(np.float32)

print("Preprocessed tensor:", X_np.shape)

class WindowDataset(Dataset):
    def __init__(self, X, Y, failures, history=12, horizon=1):
        self.X, self.Y, self.failures = X, Y, failures
        self.history, self.horizon = history, horizon
        self.indices = list(range(history, len(X)-horizon+1))
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        t = self.indices[idx]
        x = self.X[t-self.history:t]                 # H,N,F
        y = self.Y[t:t+self.horizon]                # horizon,N,4
        f = self.failures[t:t+self.horizon]         # horizon,N
        return torch.tensor(x), torch.tensor(y), torch.tensor(f)

window_ds = WindowDataset(
    X_np, Y_raw_np, failure_np,
    history=CONFIG["history"],
    horizon=CONFIG["prediction_horizon"]
)

print("Number of temporal windows:", len(window_ds))


## Stage 3 — Dynamic Reliability-Aware Node Representation (DRNR)

In [ ]:

# ============================================================
# CELL 5 — DRNR
# ============================================================
class DRNR(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.temporal = nn.GRU(in_dim, hidden, batch_first=True)
        self.energy_proj = nn.Linear(in_dim, hidden)
        self.reliability_proj = nn.Linear(in_dim, hidden)
        self.fusion = nn.Sequential(
            nn.Linear(hidden*3, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        self.failure_head = nn.Sequential(
            nn.Linear(hidden, hidden//2), nn.GELU(),
            nn.Linear(hidden//2, 1), nn.Sigmoid()
        )

    def forward(self, x):
        # x: B,H,N,F
        B,H,N,F = x.shape
        z = x.permute(0,2,1,3).reshape(B*N,H,F)
        h, _ = self.temporal(z)
        ht = h[:,-1]
        e = self.energy_proj(z[:,-1])
        r = self.reliability_proj(z[:,-1])
        node = self.fusion(torch.cat([ht,e,r], dim=-1))
        risk = self.failure_head(node)
        return node.reshape(B,N,-1), risk.reshape(B,N)

drnr = DRNR(Fdim, CONFIG["hidden_dim"]).to(DEVICE)
print(drnr)


## Stage 4–5 — Dynamic Weighted Hypergraph + RA-HAN

In [ ]:

# ============================================================
# CELL 6 — DYNAMIC HYPERGRAPH CONSTRUCTION
# ============================================================
def build_dynamic_hypergraph(node_embeddings, coords, traffic, link_quality, k=5):
    # node_embeddings: [N,D]
    Nn = node_embeddings.shape[0]

    # Spatial groups.
    n_clusters = max(2, min(Nn//3, Nn//2))
    km = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
    labels = km.fit_predict(coords)

    hyperedges = []
    weights = []

    for c in range(n_clusters):
        members = np.where(labels == c)[0].tolist()
        if len(members) >= 2:
            hyperedges.append(members)
            weights.append(float(np.mean(link_quality[members]) *
                                 (1 + np.mean(traffic[members]))))

    # Communication neighborhoods.
    dist = np.linalg.norm(coords[:,None,:] - coords[None,:,:], axis=-1)
    for i in range(Nn):
        members = np.where(dist[i] <= CONFIG["radio_range"])[0].tolist()
        if len(members) >= 2:
            members = sorted(members, key=lambda j: dist[i,j])[:k]
            hyperedges.append(members)
            weights.append(float(np.mean(link_quality[members]) *
                                 np.exp(-np.mean(dist[i,members])/100)))

    # Deduplicate.
    unique = {}
    for e,w in zip(hyperedges,weights):
        key = tuple(sorted(set(e)))
        if len(key) >= 2:
            unique[key] = max(unique.get(key, 0), w)

    hyperedges = [list(k) for k in unique.keys()]
    weights = np.array(list(unique.values()), dtype=np.float32)
    weights = weights / (weights.mean() + 1e-8)

    H = np.zeros((len(hyperedges), Nn), dtype=np.float32)
    for e_idx, e in enumerate(hyperedges):
        H[e_idx, e] = 1.0

    return H, weights, hyperedges

class HypergraphAttentionLayer(nn.Module):
    def __init__(self, dim, heads=4, dropout=0.1):
        super().__init__()
        assert dim % heads == 0
        self.dim, self.heads, self.dh = dim, heads, dim//heads
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.edge_proj = nn.Linear(dim, dim)
        self.out = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, X, Hmat, edge_weights):
        # X: B,N,D; Hmat: E,N
        B,N,D = X.shape
        E = Hmat.shape[0]

        edge_count = Hmat.sum(1, keepdims=True) + 1e-8
        edge_emb = torch.einsum("en,bnd->bed", torch.tensor(Hmat, device=X.device)) / torch.tensor(edge_count, device=X.device).view(1,E,1)
        edge_emb = self.edge_proj(edge_emb)

        q = self.q(X).view(B,N,self.heads,self.dh).transpose(1,2)
        k_edge = self.k(edge_emb).view(B,E,self.heads,self.dh).permute(0,2,1,3)
        v_edge = self.v(edge_emb).view(B,E,self.heads,self.dh).permute(0,2,1,3)

        # Node -> hyperedge attention.
        scores = torch.einsum("bhnd,bhed->bhne", q, k_edge) / math.sqrt(self.dh)
        mask = torch.tensor(Hmat, device=X.device).bool()[None,None,:,:]
        scores = scores.masked_fill(~mask, -1e9)
        alpha = F.softmax(scores, dim=-1)
        alpha = self.dropout(alpha)

        msg = torch.einsum("bhne,bhed->bhnd", alpha, v_edge)
        msg = msg.transpose(1,2).contiguous().view(B,N,D)

        # Edge-aware residual weighting.
        ew = torch.tensor(edge_weights, dtype=X.dtype, device=X.device)
        edge_strength = torch.einsum("ne,e->n", torch.tensor(Hmat, device=X.device), ew)
        edge_strength = edge_strength / (torch.tensor(Hmat, device=X.device).sum(0)+1e-8)
        msg = msg * (1 + edge_strength.view(1,N,1))

        return self.norm(X + self.out(msg))

class RAHAN(nn.Module):
    def __init__(self, dim=64, heads=4, dropout=0.1):
        super().__init__()
        self.att1 = HypergraphAttentionLayer(dim, heads, dropout)
        self.att2 = HypergraphAttentionLayer(dim, heads, dropout)
        self.cross = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim*2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(dim*2, dim)
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, X, Hmat, edge_weights):
        z = self.att1(X, Hmat, edge_weights)
        z = self.att2(z, Hmat, edge_weights)
        cross, _ = self.cross(z,z,z)
        return self.norm(z + cross + self.ffn(z))

print("RA-HAN components created.")


## Stage 6 — Mamba-2 Temporal Prediction

In [ ]:

# ============================================================
# CELL 7 — CPU-SAFE MAMBA-2-STYLE SELECTIVE STATE SPACE MODEL
# ============================================================
class Mamba2StyleBlock(nn.Module):
    """
    Lightweight selective state-space block inspired by the
    Mamba-2 design philosophy. It is implemented using standard
    PyTorch operators so it runs on CPU-only Colab environments.

    It provides:
      - input-dependent delta/gating
      - recurrent state update
      - causal temporal processing
      - residual + normalization
    """
    def __init__(self, d_model, d_state=16, expand=2):
        super().__init__()
        self.d_model = d_model
        self.inner = d_model * expand
        self.in_proj = nn.Linear(d_model, self.inner*2)
        self.delta = nn.Linear(self.inner, self.inner)
        self.B = nn.Linear(self.inner, d_state)
        self.C = nn.Linear(self.inner, d_state)
        self.A_log = nn.Parameter(torch.randn(self.inner, d_state) * 0.05)
        self.out_proj = nn.Linear(self.inner, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: B,T,D
        residual = x
        x = self.norm(x)
        uv = self.in_proj(x)
        u, gate = uv.chunk(2, dim=-1)
        gate = torch.sigmoid(gate)

        B,T,I = u.shape
        state = torch.zeros(B,I,self.A_log.shape[1], device=x.device, dtype=x.dtype)
        outputs = []

        A = -torch.exp(self.A_log)

        for t in range(T):
            ut = u[:,t]
            dt = F.softplus(self.delta(ut)) + 1e-3
            Bt = self.B(ut).unsqueeze(1)
            Ct = self.C(ut).unsqueeze(1)

            # Stable discretized state transition.
            decay = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0))
            state = decay * state + Bt * ut.unsqueeze(-1)
            yt = (Ct * state).sum(-1)
            yt = yt * gate[:,t]
            outputs.append(yt)

        y = torch.stack(outputs, dim=1)
        return residual + self.out_proj(y)

class TemporalPredictor(nn.Module):
    def __init__(self, node_dim, temporal_dim=64):
        super().__init__()
        self.input = nn.Linear(node_dim, temporal_dim)
        self.mamba1 = Mamba2StyleBlock(temporal_dim)
        self.mamba2 = Mamba2StyleBlock(temporal_dim)
        self.temporal_attn = nn.MultiheadAttention(temporal_dim, 4, batch_first=True)
        self.energy_head = nn.Linear(temporal_dim, 2)  # residual energy, depletion rate
        self.rel_head = nn.Linear(temporal_dim, 2)     # failure probability, reliability

    def forward(self, x):
        # x: B,T,N,D -> predict each node independently
        B,T,N,D = x.shape
        z = x.permute(0,2,1,3).reshape(B*N,T,D)
        z = self.input(z)
        z = self.mamba1(z)
        z = self.mamba2(z)
        a,_ = self.temporal_attn(z,z,z)
        h = a[:,-1]
        out = torch.cat([self.energy_head(h), self.rel_head(h)], dim=-1)
        return out.reshape(B,N,4)

print("Mamba-2-style predictor ready.")


## Prediction Dataset + Training

In [ ]:

# ============================================================
# CELL 8 — FEATURE ENCODER + TEMPORAL TRAINING
# ============================================================
# For the temporal predictor, we create deterministic DRNR/RA-HAN
# embeddings from each window. To keep the notebook fully runnable,
# the representation encoder is trained jointly with a prediction head.

class FullPredictor(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.drnr = DRNR(in_dim, hidden)
        self.rahan = RAHAN(hidden, CONFIG["n_heads"], CONFIG["dropout"])
        self.temporal = TemporalPredictor(hidden, CONFIG["temporal_dim"])

    def encode_last(self, x):
        # x: B,H,N,F
        node, risk = self.drnr(x)
        return node, risk

    def forward(self, x, Hmats, Eweights):
        # Build historical embeddings for each timestep.
        B,H,N,Fd = x.shape
        embeddings = []
        risks = []
        for t in range(H):
            node, risk = self.drnr(x[:, :t+1])
            # Same dynamic hypergraph for batch based on topology proxy.
            embeddings.append(node)
            risks.append(risk)
        hist = torch.stack(embeddings, dim=1)  # B,H,N,D
        z = []
        for t in range(H):
            z.append(self.rahan(hist[:,t], Hmats[t], Eweights[t]))
        z = torch.stack(z, dim=1)
        pred = self.temporal(z)
        return pred, z, torch.stack(risks, dim=1)

# Create topology hypergraphs for each historical time using raw unscaled signals.
Hmats, Eweights = [], []
for t in range(CONFIG["n_steps"]):
    # Use the node embeddings from a deterministic projection as a construction proxy.
    raw_node = torch.tensor(X_np[t], dtype=torch.float32)
    traffic = raw_df[raw_df.time==t]["traffic_load"].values
    lq = raw_df[raw_df.time==t]["link_quality"].values
    Hm, Ew, _ = build_dynamic_hypergraph(raw_node.numpy(), coordinates, traffic, lq, CONFIG["hyperedge_k"])
    Hmats.append(torch.tensor(Hm, dtype=torch.float32, device=DEVICE))
    Eweights.append(torch.tensor(Ew, dtype=torch.float32, device=DEVICE))

# Use a fixed compact topology from the final training snapshot for efficient batching.
H_train = Hmats[CONFIG["history"]-1]
E_train = Eweights[CONFIG["history"]-1]

pred_ds = window_ds
loader = DataLoader(pred_ds, batch_size=CONFIG["batch_size"], shuffle=True)

model = FullPredictor(Fdim, CONFIG["hidden_dim"]).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

def prediction_loss(pred, target):
    # target: B,horizon,N,4; use first horizon as primary supervised target.
    y = target[:,0]
    # scale targets to numerically stable ranges
    y_scaled = y.clone()
    y_scaled[:,:,0] /= CONFIG["initial_energy"]
    y_scaled[:,:,1] /= 0.5
    y_scaled[:,:,2] = torch.clamp(y_scaled[:,:,2],0,1)
    y_scaled[:,:,3] = torch.clamp(y_scaled[:,:,3],0,1)

    p = pred
    return (
        F.mse_loss(torch.sigmoid(p[:,:,0]), y_scaled[:,:,0]) +
        F.mse_loss(torch.sigmoid(p[:,:,1]), torch.clamp(y_scaled[:,:,1],0,1)) +
        F.binary_cross_entropy(torch.sigmoid(p[:,:,2]), y_scaled[:,:,2]) +
        F.mse_loss(torch.sigmoid(p[:,:,3]), y_scaled[:,:,3])
    )

model.train()
history_loss = []

for epoch in range(CONFIG["epochs_prediction"]):
    total = 0.0
    nbat = 0
    for xb, yb, fb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()

        # Lightweight batching: reuse one dynamic topology for the sequence.
        B,H,N,Fd = xb.shape
        embeddings = []
        for tt in range(H):
            node,_ = model.drnr(xb[:, :tt+1])
            node = model.rahan(node, H_train, E_train)
            embeddings.append(node)
        hist_emb = torch.stack(embeddings, dim=1)

        pred = model.temporal(hist_emb)
        loss = prediction_loss(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total += loss.item()
        nbat += 1

    history_loss.append(total/max(nbat,1))
    print(f"Epoch {epoch+1:02d}/{CONFIG['epochs_prediction']} - loss={history_loss[-1]:.5f}")

torch.save(model.state_dict(), os.path.join(OUT_DIR, "RA_HAN_Mamba2_predictor.pth"))
print("Saved predictor.")


## Mamba-2 Prediction Evaluation

In [ ]:

# ============================================================
# CELL 9 — PREDICTION METRICS
# ============================================================
model.eval()
predictions, actuals, fail_actual = [], [], []

with torch.no_grad():
    for xb, yb, fb in DataLoader(pred_ds, batch_size=CONFIG["batch_size"], shuffle=False):
        xb = xb.to(DEVICE)
        B,H,N,Fd = xb.shape
        embeddings=[]
        for tt in range(H):
            node,_ = model.drnr(xb[:, :tt+1])
            node = model.rahan(node, H_train, E_train)
            embeddings.append(node)
        hist_emb = torch.stack(embeddings, dim=1)
        pred = model.temporal(hist_emb)
        pred_np = pred.cpu().numpy()
        # inverse-like mapping
        p = np.zeros_like(pred_np)
        p[:,:,0] = 100 * (1/(1+np.exp(-pred_np[:,:,0])))
        p[:,:,1] = 0.5 * (1/(1+np.exp(-pred_np[:,:,1])))
        p[:,:,2] = 1/(1+np.exp(-pred_np[:,:,2]))
        p[:,:,3] = 1/(1+np.exp(-pred_np[:,:,3]))
        predictions.append(p)
        actuals.append(yb[:,0].numpy())
        fail_actual.append(fb[:,0].numpy())

pred_arr = np.concatenate(predictions, axis=0)
act_arr = np.concatenate(actuals, axis=0)
fail_arr = np.concatenate(fail_actual, axis=0)

names = ["Residual Energy", "Energy Depletion Rate", "Failure Probability", "Reliability"]
metrics_rows = []

for j,name in enumerate(names):
    ytrue = act_arr[:,:,j].ravel()
    ypred = pred_arr[:,:,j].ravel()
    mae = mean_absolute_error(ytrue, ypred)
    rmse = np.sqrt(mean_squared_error(ytrue, ypred))
    denom = np.where(np.abs(ytrue)<1e-5, 1e-5, np.abs(ytrue))
    mape = np.mean(np.abs((ytrue-ypred)/denom))*100
    r2 = r2_score(ytrue, ypred)
    metrics_rows.append([name, mae, rmse, mape, r2])

prediction_metrics = pd.DataFrame(
    metrics_rows, columns=["Target","MAE","RMSE","MAPE_percent","R2"]
)
display(prediction_metrics)
prediction_metrics.to_csv(os.path.join(OUT_DIR,"mamba2_prediction_metrics.csv"), index=False)

# Failure classification
true_fail = (act_arr[:,:,2] > 0.5).astype(int).ravel()
pred_fail = (pred_arr[:,:,2] > 0.5).astype(int).ravel()

failure_metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(true_fail,pred_fail),
    "Precision": precision_score(true_fail,pred_fail,zero_division=0),
    "Recall": recall_score(true_fail,pred_fail,zero_division=0),
    "F1": f1_score(true_fail,pred_fail,zero_division=0)
}])
display(failure_metrics)
failure_metrics.to_csv(os.path.join(OUT_DIR,"failure_prediction_metrics.csv"), index=False)


In [ ]:

# ============================================================
# CELL 10 — REQUIRED MAMBA-2 PLOTS
# ============================================================
plt.figure(figsize=(11,5))
nplot = min(300, len(act_arr.ravel()))
plt.plot(act_arr[:,:,0].ravel()[:nplot], label="Actual")
plt.plot(pred_arr[:,:,0].ravel()[:nplot], label="Predicted")
plt.title("Actual vs Predicted Residual Energy", fontweight="bold")
plt.xlabel("Node-window sample"); plt.ylabel("Residual Energy")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"actual_vs_predicted_residual_energy.png"), dpi=300)
plt.show()

plt.figure(figsize=(11,5))
plt.plot(act_arr[:,:,1].ravel()[:nplot], label="Actual")
plt.plot(pred_arr[:,:,1].ravel()[:nplot], label="Predicted")
plt.title("Actual vs Predicted Energy Depletion Rate", fontweight="bold")
plt.xlabel("Node-window sample"); plt.ylabel("Energy depletion rate")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"actual_vs_predicted_depletion.png"), dpi=300)
plt.show()

plt.figure(figsize=(11,5))
plt.plot(act_arr[:,:,2].ravel()[:nplot], label="Actual")
plt.plot(pred_arr[:,:,2].ravel()[:nplot], label="Predicted")
plt.title("Actual vs Predicted Failure Probability", fontweight="bold")
plt.xlabel("Node-window sample"); plt.ylabel("Failure probability")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"actual_vs_predicted_failure_probability.png"), dpi=300)
plt.show()

cm = confusion_matrix(true_fail,pred_fail)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Node Failure Prediction Confusion Matrix", fontweight="bold")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"failure_confusion_matrix.png"), dpi=300)
plt.show()


## Stage 7–8 — RAMS: PPO + Centralized Training / Decentralized Execution

In [ ]:

# ============================================================
# CELL 11 — RAMS ENVIRONMENT
# ============================================================
class RAMS_WSN_Env:
    def __init__(self, coords, init_energy, distance_matrix, seed=42):
        self.coords = coords
        self.init_energy = init_energy.copy()
        self.D = distance_matrix.copy()
        self.N = len(coords)
        self.rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.energy = self.init_energy.copy()
        self.alive = np.ones(self.N, dtype=np.float32)
        self.prev_active = np.ones(self.N, dtype=np.float32)
        self.t = 0
        self.total_delivered = 0
        self.total_sent = 0
        return self.get_state()

    def get_state(self):
        dist_sink = np.linalg.norm(self.coords - np.array([50,50]), axis=1)
        traffic = np.clip(0.06*(1+0.3*np.sin(self.t/10)) *
                          self.rng.lognormal(0,0.12,self.N), .005, .30)
        reliability = np.clip(
            0.95*np.exp(-dist_sink/180)*(0.8+0.2*self.alive)
            - 0.18*(traffic) + self.rng.normal(0,.02,self.N),
            .05,.995
        )
        risk = np.clip(
            0.02 + .60*(1-self.energy/self.init_energy) + .35*(1-reliability),
            0,.99
        )
        return np.stack([
            self.energy/self.init_energy,
            traffic,
            reliability,
            risk,
            self.prev_active,
            dist_sink/100
        ], axis=1).astype(np.float32)

    def step(self, actions):
        # actions: 0=sleep, 1=active/reliable transmission
        state_before = self.get_state()
        energy_norm = self.energy/self.init_energy
        reliability = state_before[:,2]
        risk = state_before[:,3]
        traffic = state_before[:,1]

        active = ((actions == 1) & (self.alive > 0)).astype(np.float32)

        # Routing/transmission cost. High-risk nodes cost more due to retransmissions.
        base_cost = 0.06 + 0.025*traffic + 0.025*(1-reliability) + 0.015*state_before[:,5]
        consumption = active * base_cost * (1 + 0.6*risk)
        self.energy = np.maximum(self.energy - consumption, 0)

        # Delivery probability.
        delivery_prob = np.clip(reliability * (1 - 0.25*risk), .01, .99)
        sent = active * (traffic > 0.02)
        delivered = sent * (self.rng.random(self.N) < delivery_prob)

        # Random failure.
        failure_p = np.clip(0.005 + 0.06*risk + 0.02*(1-reliability), 0, .5)
        failed = (self.rng.random(self.N) < failure_p).astype(np.float32)
        self.alive = self.alive * (self.energy > 0).astype(np.float32) * (1-failed)

        # Reward: lifetime + reliability + delivery + energy balance - cost - risk.
        avg_energy = np.mean(self.energy/self.init_energy)
        energy_balance = np.std(self.energy/self.init_energy)
        reward_vec = (
            2.0*delivered
            + 1.5*active*reliability
            + 1.0*avg_energy
            - 1.8*consumption
            - 2.0*active*risk
            - 0.8*energy_balance
        )

        network_reward = float(np.mean(reward_vec) + 2.0*np.mean(self.alive))
        self.total_delivered += delivered.sum()
        self.total_sent += sent.sum()
        self.prev_active = active
        self.t += 1

        done = (self.t >= CONFIG["max_episode_steps"]) or (self.alive.sum() <= 1)
        info = {
            "alive": self.alive.sum(),
            "avg_energy": self.energy.mean(),
            "delivery_ratio": self.total_delivered/max(self.total_sent,1),
            "energy_consumption": consumption.sum()
        }
        return self.get_state(), network_reward, done, info

env = RAMS_WSN_Env(coordinates, np.full(CONFIG["n_nodes"], CONFIG["initial_energy"],dtype=np.float32), distance_matrix)
print("RAMS environment ready. Agents:", env.N)


In [ ]:

# ============================================================
# CELL 12 — CENTRALIZED-TRAINING / DECENTRALIZED-EXECUTION PPO
# ============================================================
class MultiAgentActorCritic(nn.Module):
    def __init__(self, obs_dim, n_agents, hidden=128):
        super().__init__()
        self.n_agents = n_agents
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2)
        )
        # Centralized critic sees all agents' observations.
        self.critic = nn.Sequential(
            nn.Linear(obs_dim*n_agents, hidden*2), nn.Tanh(),
            nn.Linear(hidden*2, hidden), nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def actor_dist(self, obs):
        logits = self.actor(obs)
        return Categorical(logits=logits)

    def value(self, global_obs):
        return self.critic(global_obs)

agent_model = MultiAgentActorCritic(6, env.N).to(DEVICE)
ppo_optimizer = torch.optim.Adam(agent_model.parameters(), lr=CONFIG["ppo_lr"])

def collect_rollout(model, env, steps=64):
    obs = env.reset()
    batch = []
    for _ in range(steps):
        obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            dist = model.actor_dist(obs_t)
            action = dist.sample()
            logp = dist.log_prob(action)
            global_obs = obs_t.reshape(1,-1)
            value = model.value(global_obs).squeeze()

        next_obs, reward, done, info = env.step(action.cpu().numpy())

        batch.append({
            "obs": obs.copy(),
            "global_obs": obs.copy().reshape(-1),
            "actions": action.cpu().numpy(),
            "logp": logp.cpu().numpy(),
            "reward": reward,
            "value": value.item(),
            "done": done,
            "next_obs": next_obs.copy()
        })

        obs = env.reset() if done else next_obs
    return batch

def compute_gae(batch, gamma=0.99, lam=0.95):
    rewards = np.array([b["reward"] for b in batch], dtype=np.float32)
    values = np.array([b["value"] for b in batch] + [0], dtype=np.float32)
    dones = np.array([b["done"] for b in batch], dtype=np.float32)

    adv = np.zeros(len(batch), dtype=np.float32)
    gae = 0
    for t in reversed(range(len(batch))):
        delta = rewards[t] + gamma*values[t+1]*(1-dones[t]) - values[t]
        gae = delta + gamma*lam*(1-dones[t])*gae
        adv[t] = gae
    returns = adv + values[:-1]
    return adv, returns

ppo_history = []

for epoch in range(CONFIG["epochs_ppo"]):
    batch = collect_rollout(agent_model, env, CONFIG["ppo_steps_per_epoch"])
    adv, returns = compute_gae(batch, CONFIG["gamma"], CONFIG["gae_lambda"])

    obs_all = torch.tensor(np.concatenate([b["obs"] for b in batch]), dtype=torch.float32, device=DEVICE)
    actions_all = torch.tensor(np.concatenate([b["actions"] for b in batch]), dtype=torch.long, device=DEVICE)
    old_logp_all = torch.tensor(np.concatenate([b["logp"] for b in batch]), dtype=torch.float32, device=DEVICE)
    global_all = obs_all.reshape(len(batch), -1)
    adv_t = torch.tensor(np.repeat(adv, env.N), dtype=torch.float32, device=DEVICE)
    ret_t = torch.tensor(np.repeat(returns, env.N), dtype=torch.float32, device=DEVICE)

    # Normalize advantages.
    adv_t = (adv_t - adv_t.mean())/(adv_t.std()+1e-8)

    for _ in range(4):
        dist = agent_model.actor_dist(obs_all)
        logp = dist.log_prob(actions_all)
        entropy = dist.entropy().mean()
        ratio = torch.exp(logp - old_logp_all)

        policy_loss = -torch.min(
            ratio*adv_t,
            torch.clamp(ratio, 1-CONFIG["ppo_clip"], 1+CONFIG["ppo_clip"])*adv_t
        ).mean()

        values = agent_model.value(global_all).squeeze()
        value_loss = F.mse_loss(values, ret_t)
        loss = policy_loss + 0.5*value_loss - 0.01*entropy

        ppo_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(agent_model.parameters(), 0.5)
        ppo_optimizer.step()

    ppo_history.append([epoch+1, float(loss.item()), float(np.mean([b["reward"] for b in batch]))])
    print(f"PPO epoch {epoch+1:02d}/{CONFIG['epochs_ppo']} - loss={loss.item():.4f}, reward={ppo_history[-1][2]:.4f}")

torch.save(agent_model.state_dict(), os.path.join(OUT_DIR,"RAMS_PPO_CTDE.pth"))
pd.DataFrame(ppo_history, columns=["Epoch","Loss","MeanReward"]).to_csv(
    os.path.join(OUT_DIR,"ppo_training_history.csv"), index=False
)


## Stage 8–9 — Closed-Loop Network Updating and Lifetime Simulation

In [ ]:

# ============================================================
# CELL 13 — CLOSED-LOOP POLICY SIMULATION
# ============================================================
def run_policy(model, env, steps=120, risk_aware=True):
    obs = env.reset()
    records = []

    for t in range(steps):
        obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            dist = model.actor_dist(obs_t)
            if risk_aware:
                action = dist.probs.argmax(dim=-1)
            else:
                # Baseline heuristic: activate nodes with highest energy.
                energy = obs[:,0]
                threshold = np.quantile(energy, 0.5)
                action = torch.tensor((energy >= threshold).astype(int), device=DEVICE)

        next_obs, reward, done, info = env.step(action.cpu().numpy())

        records.append({
            "time": t,
            "alive_nodes": info["alive"],
            "dead_nodes": env.N-info["alive"],
            "average_residual_energy": info["avg_energy"],
            "delivery_ratio": info["delivery_ratio"],
            "energy_consumption": info["energy_consumption"],
            "reward": reward
        })

        obs = next_obs
        if done:
            break

    return pd.DataFrame(records)

closed_loop_df = run_policy(agent_model, env, CONFIG["max_episode_steps"], True)
closed_loop_df.to_csv(os.path.join(OUT_DIR,"closed_loop_rams_results.csv"), index=False)

display(closed_loop_df.head())

plt.figure(figsize=(10,5))
plt.plot(closed_loop_df["time"], closed_loop_df["alive_nodes"])
plt.title("RAMS Closed-Loop Alive Nodes", fontweight="bold")
plt.xlabel("Scheduling interval"); plt.ylabel("Alive nodes")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"rams_alive_nodes.png"), dpi=300)
plt.show()

plt.figure(figsize=(10,5))
plt.plot(closed_loop_df["time"], closed_loop_df["average_residual_energy"])
plt.title("RAMS Average Residual Energy", fontweight="bold")
plt.xlabel("Scheduling interval"); plt.ylabel("Average residual energy")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"rams_average_energy.png"), dpi=300)
plt.show()


## Stage 10 — Lifetime, Reliability, Energy and Complexity Evaluation

In [ ]:

# ============================================================
# CELL 14 — EVALUATION METRICS
# ============================================================
def lifetime_metrics(result, n_nodes):
    alive = result["alive_nodes"].values
    times = result["time"].values

    fnd = next((times[i] for i,v in enumerate(alive) if v < n_nodes), np.nan)
    hnd = next((times[i] for i,v in enumerate(alive) if v <= n_nodes/2), np.nan)
    lnd = next((times[i] for i,v in enumerate(alive) if v <= 1), np.nan)

    return {
        "First_Node_Death": fnd,
        "Half_Node_Death": hnd,
        "Last_Node_Death": lnd,
        "Average_Residual_Energy": result["average_residual_energy"].mean(),
        "Energy_Consumption_Total": result["energy_consumption"].sum(),
        "Packet_Delivery_Ratio": result["delivery_ratio"].iloc[-1],
        "Average_Reward": result["reward"].mean(),
        "Dead_Nodes_Final": result["dead_nodes"].iloc[-1]
    }

rams_metrics = lifetime_metrics(closed_loop_df, env.N)
display(pd.DataFrame([rams_metrics]))
pd.DataFrame([rams_metrics]).to_csv(os.path.join(OUT_DIR,"rams_lifetime_metrics.csv"), index=False)

# Computational cost / inference timing.
sample_obs = env.reset()
start = time.perf_counter()
for _ in range(100):
    with torch.no_grad():
        _ = agent_model.actor_dist(torch.tensor(sample_obs,dtype=torch.float32,device=DEVICE)).probs
inference_ms = (time.perf_counter()-start)*1000/100

complexity = pd.DataFrame([{
    "N_nodes": env.N,
    "RAHAN_hidden_dim": CONFIG["hidden_dim"],
    "Attention_heads": CONFIG["n_heads"],
    "PPO_parameters": sum(p.numel() for p in agent_model.parameters()),
    "Predictor_parameters": sum(p.numel() for p in model.parameters()),
    "PPO_inference_ms": inference_ms,
    "Device": str(DEVICE)
}])
display(complexity)
complexity.to_csv(os.path.join(OUT_DIR,"computational_complexity.csv"), index=False)


## Baselines and Ablation Study

In [ ]:

# ============================================================
# CELL 15 — BASELINE SCHEDULERS
# ============================================================
def run_heuristic(env, policy_name, steps=120):
    obs = env.reset()
    rec=[]
    for t in range(steps):
        energy = obs[:,0]
        rel = obs[:,2]
        risk = obs[:,3]
        traffic = obs[:,1]

        if policy_name == "All_Active":
            actions = np.ones(env.N,dtype=int)
        elif policy_name == "Energy_Greedy":
            actions = (energy >= np.quantile(energy, .5)).astype(int)
        elif policy_name == "Reliability_Greedy":
            actions = (rel >= np.quantile(rel, .5)).astype(int)
        elif policy_name == "Risk_Aware_Heuristic":
            score = .45*energy + .40*rel - .35*risk + .10*traffic
            actions = (score >= np.quantile(score,.5)).astype(int)
        else:
            actions = np.ones(env.N,dtype=int)

        obs, reward, done, info = env.step(actions)
        rec.append({
            "time":t, "alive_nodes":info["alive"],
            "dead_nodes":env.N-info["alive"],
            "average_residual_energy":info["avg_energy"],
            "delivery_ratio":info["delivery_ratio"],
            "energy_consumption":info["energy_consumption"],
            "reward":reward
        })
        if done: break
    return pd.DataFrame(rec)

baseline_names = [
    "All_Active", "Energy_Greedy",
    "Reliability_Greedy", "Risk_Aware_Heuristic"
]

baseline_results = {}
rows=[]
for name in baseline_names:
    rr = run_heuristic(env, name, CONFIG["max_episode_steps"])
    baseline_results[name] = rr
    mm = lifetime_metrics(rr, env.N)
    mm["Method"] = name
    rows.append(mm)

full_m = rams_metrics.copy()
full_m["Method"] = "RA-HAN + Mamba-2 + RAMS-PPO"
rows.append(full_m)

comparison = pd.DataFrame(rows)[[
    "Method","First_Node_Death","Half_Node_Death","Last_Node_Death",
    "Average_Residual_Energy","Energy_Consumption_Total",
    "Packet_Delivery_Ratio","Dead_Nodes_Final"
]]

display(comparison)
comparison.to_csv(os.path.join(OUT_DIR,"baseline_comparison.csv"), index=False)


In [ ]:

# ============================================================
# CELL 16 — ABLATION EXPERIMENTS
# ============================================================
# Practical ablations:
# 1. Without DRNR: raw node observations
# 2. Without RA-HAN: local/mean representation
# 3. Without Mamba-2: last-state predictor
# 4. Without PPO: heuristic scheduling
#
# For fair operational comparison, the same simulator seed is used.

def run_ablation(ablation, steps=120, seed=100):
    local_env = RAMS_WSN_Env(
        coordinates,
        np.full(CONFIG["n_nodes"], CONFIG["initial_energy"],dtype=np.float32),
        distance_matrix,
        seed=seed
    )
    obs = local_env.reset()
    rec=[]

    for t in range(steps):
        energy, traffic, rel, risk, prev, dist = obs.T

        if ablation == "Without_DRNR":
            score = .70*energy + .30*rel
        elif ablation == "Without_RAHAN":
            score = .55*energy + .45*rel - .20*risk
        elif ablation == "Without_Mamba2":
            # No future prediction: rely only on current risk.
            score = .50*energy + .50*rel - .50*risk
        elif ablation == "Without_PPO":
            score = energy
        else:
            score = .45*energy + .40*rel - .35*risk + .10*traffic

        actions = (score >= np.quantile(score, .5)).astype(int)

        obs, reward, done, info = local_env.step(actions)
        rec.append({
            "time":t, "alive_nodes":info["alive"],
            "dead_nodes":local_env.N-info["alive"],
            "average_residual_energy":info["avg_energy"],
            "delivery_ratio":info["delivery_ratio"],
            "energy_consumption":info["energy_consumption"],
            "reward":reward
        })
        if done: break

    return pd.DataFrame(rec)

ablation_names = ["Without_DRNR","Without_RAHAN","Without_Mamba2","Without_PPO"]
ablation_rows=[]

for name in ablation_names:
    rr = run_ablation(name, CONFIG["max_episode_steps"], SEED)
    mm = lifetime_metrics(rr, CONFIG["n_nodes"])
    mm["Method"] = name
    ablation_rows.append(mm)

mm = rams_metrics.copy()
mm["Method"] = "Full_RA-HAN_Mamba2_RAMS"
ablation_rows.append(mm)

ablation_table = pd.DataFrame(ablation_rows)
display(ablation_table)
ablation_table.to_csv(os.path.join(OUT_DIR,"ablation_results.csv"), index=False)


## Final Comparison Plots + Export

In [ ]:

# ============================================================
# CELL 17 — FINAL COMPARISON VISUALIZATION
# ============================================================
all_methods = pd.concat([
    comparison.assign(Type="Baseline/Full"),
    ablation_table[[
        "Method","First_Node_Death","Half_Node_Death","Last_Node_Death",
        "Average_Residual_Energy","Energy_Consumption_Total",
        "Packet_Delivery_Ratio","Dead_Nodes_Final"
    ]].assign(Type="Ablation")
], ignore_index=True)

plt.figure(figsize=(12,6))
plot_df = all_methods.sort_values("First_Node_Death", ascending=False)
sns.barplot(data=plot_df, x="First_Node_Death", y="Method")
plt.title("First Node Death — Lifetime Comparison", fontweight="bold")
plt.xlabel("Scheduling interval"); plt.ylabel("")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"first_node_death_comparison.png"), dpi=300)
plt.show()

plt.figure(figsize=(12,6))
sns.barplot(data=plot_df, x="Packet_Delivery_Ratio", y="Method")
plt.title("Packet Delivery Ratio Comparison", fontweight="bold")
plt.xlabel("Packet Delivery Ratio"); plt.ylabel("")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"pdr_comparison.png"), dpi=300)
plt.show()

plt.figure(figsize=(12,6))
sns.barplot(data=plot_df, x="Average_Residual_Energy", y="Method")
plt.title("Average Residual Energy Comparison", fontweight="bold")
plt.xlabel("Average residual energy"); plt.ylabel("")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"energy_comparison.png"), dpi=300)
plt.show()

# Save all important tables into one Excel workbook.
excel_path = os.path.join(OUT_DIR, "RA_HAN_RAMS_results.xlsx")
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    raw_df.head(5000).to_excel(writer, sheet_name="Raw_Sample", index=False)
    prediction_metrics.to_excel(writer, sheet_name="Mamba2_Metrics", index=False)
    failure_metrics.to_excel(writer, sheet_name="Failure_Metrics", index=False)
    comparison.to_excel(writer, sheet_name="Baselines", index=False)
    ablation_table.to_excel(writer, sheet_name="Ablation", index=False)
    complexity.to_excel(writer, sheet_name="Complexity", index=False)
    closed_loop_df.to_excel(writer, sheet_name="RAMS_ClosedLoop", index=False)

print("Excel:", excel_path)
print("Output directory:", OUT_DIR)



## Stage 10 — Interpretation Template

The generated tables support the manuscript's evaluation claims:

### Prediction
- **MAE / RMSE / MAPE / R²** for residual energy
- **MAE / RMSE / MAPE / R²** for energy depletion
- **MAE / RMSE / MAPE / R²** for failure probability
- **MAE / RMSE / MAPE / R²** for communication reliability
- **Accuracy / Precision / Recall / F1** for node failure classification
- Confusion matrix

### Network lifetime
- First Node Death (FND)
- Half Node Death (HND)
- Last Node Death (LND)
- Average residual energy
- Total energy consumption
- Packet delivery ratio
- Dead-node count
- Reward / reliability

### Ablation
- Without DRNR → contribution of dynamic reliability-aware representation
- Without RA-HAN → contribution of weighted hypergraph attention
- Without Mamba-2 → contribution of temporal future-state prediction
- Without PPO → contribution of adaptive risk-aware multi-agent scheduling
- Full model → complete RA-HAN + Mamba-2 + RAMS-PPO framework

### Important note for real experiments
For publication-grade results, replace the synthetic Stage-1 simulator with synchronized **NS-3 / OMNeT++ traces or a real WSN dataset**, preserving the same feature names. The remaining stages can then be reused with minimal changes.


In [ ]:

# ============================================================
# CELL 18 — PACKAGE OUTPUTS INTO ZIP
# ============================================================
import shutil, os

zip_base = "/content/RA_HAN_RAMS_outputs"
zip_file = shutil.make_archive(zip_base, "zip", OUT_DIR)
print("ZIP created:", zip_file)

# List key files.
for root, dirs, files in os.walk(OUT_DIR):
    for f in sorted(files):
        print(os.path.join(root, f))
